# CryptoLens — Model Training Notebook
Interactive exploration before running the full training pipeline.

In [ ]:
import sys
sys.path.insert(0, '../backend')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from app.services.feature_engineering import build_features, create_labels, get_feature_columns
from app.services.data_collector import load_ohlcv

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (14, 6)
print('Imports OK')

In [ ]:
# Load BTC data
df = load_ohlcv('BTCUSDT', limit=500)
print(f'Loaded {len(df)} rows')
df.tail()

In [ ]:
# Build features
feat = build_features(df)
labels = create_labels(df, horizon=7, threshold=0.03)
print('Label distribution:')
print(labels.value_counts())
feat.tail()

In [ ]:
# Plot price with BUY/SELL labels
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

ax1.plot(df['timestamp'], df['close'], color='#6c63ff', linewidth=1)
buy_idx = labels[labels == 'BUY'].index
sell_idx = labels[labels == 'SELL'].index
ax1.scatter(df.loc[buy_idx, 'timestamp'], df.loc[buy_idx, 'close'], color='#00d97e', s=30, zorder=5, label='BUY')
ax1.scatter(df.loc[sell_idx, 'timestamp'], df.loc[sell_idx, 'close'], color='#ff4d6a', s=30, zorder=5, label='SELL')
ax1.set_ylabel('Price (USD)')
ax1.legend()
ax1.set_title('BTC/USDT — Labeled Signals')

ax2.plot(df['timestamp'], feat['rsi_14'], color='#f5a623')
ax2.axhline(30, color='#00d97e', linestyle='--', alpha=0.5)
ax2.axhline(70, color='#ff4d6a', linestyle='--', alpha=0.5)
ax2.set_ylabel('RSI(14)')

ax3.plot(df['timestamp'], feat['macd'], color='#4f8ef5', label='MACD')
ax3.plot(df['timestamp'], feat['macd_signal'], color='#ff4d6a', label='Signal')
ax3.bar(df['timestamp'], feat['macd_hist'], color=['#00d97e' if v > 0 else '#ff4d6a' for v in feat['macd_hist']], alpha=0.5, label='Histogram')
ax3.set_ylabel('MACD')
ax3.legend()

plt.tight_layout()
plt.savefig('../data/btc_analysis.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Quick XGBoost sanity check
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import xgboost as xgb

cols = get_feature_columns()
label_map = {'BUY': 0, 'HOLD': 1, 'SELL': 2}

X = feat[cols].dropna()
y = labels.loc[X.index].map(label_map).dropna()
X = X.loc[y.index]

# Time-based split (no shuffle!)
split = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

model = xgb.XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42)
model.fit(X_train, y_train)
preds = model.predict(X_test)

print(classification_report(y_test, preds, target_names=['BUY', 'HOLD', 'SELL']))